In [ ]:
def short_id(x):
    return str(x)[:8]

# FIX timestamp
sensor_df["date"] = pd.to_datetime(sensor_df["timestamp"], unit="s").dt.date
event_df["date"] = pd.to_datetime(event_df["recordAt"], unit="s").dt.date

# normalize label
label_df["event_id"] = label_df["event_id"].astype(str).str[:8]

In [ ]:
# %%
def build_subtracks(sensor_df, L=10):
    sub_tracks = []

    for tid, g in sensor_df.groupby("trackId"):
        g = g.sort_values("timestamp")
        xy = g[["latitude", "longitude"]].values.astype(np.float32)

        for w in range(5, 11):
            for i in range(0, len(xy) - w + 1, 2):
                seg = xy[i:i+w].copy()

                seg = normalize_xy(seg)
                seg = pad_or_truncate(seg, L)

                sub_tracks.append({
                    "track_id": tid,
                    "date": g["date"].iloc[0],   # 🔥 giữ date
                    "segment": seg
                })

    return sub_tracks


sub_tracks = build_subtracks(sensor_df)
print("Subtracks:", len(sub_tracks))

In [ ]:
# %%
track_to_sub = {}

for i, s in enumerate(sub_tracks):
    key = (s["track_id"], s["date"])

    if key not in track_to_sub:
        track_to_sub[key] = []

    track_to_sub[key].append(i)

In [ ]:
# %%
report_seqs = []
track_seqs = []
pairs = []

for _, row in label_df.iterrows():
    eid = row["event_id"]
    tid = row["track_id"]

    ev = event_df[
        event_df["event_id"].astype(str).str[:8] == eid
    ]

    if len(ev) == 0:
        continue

    for _, ev_row in ev.iterrows():
        d = ev_row["date"]

        key = (tid, d)

        if key not in track_to_sub:
            continue

        sub_idxs = track_to_sub[key]

        # build report seq 1 lần
        r_seq = encode_report_seq(ev_row)

        for sid in sub_idxs:
            t_seq = sub_tracks[sid]["segment"]

            ridx = len(report_seqs)
            tidx = len(track_seqs)

            report_seqs.append(r_seq)
            track_seqs.append(t_seq)

            pairs.append((ridx, tidx))


print("Pairs:", len(pairs))
print("Report:", len(report_seqs))
print("Track:", len(track_seqs))

In [ ]:
# %%
class PairDataset(torch.utils.data.Dataset):
    def __init__(self, report_seqs, track_seqs, pairs):
        self.r = report_seqs
        self.t = track_seqs
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        rid, tid = self.pairs[idx]

        r = torch.tensor(self.r[rid], dtype=torch.float32)
        p = torch.tensor(self.t[tid], dtype=torch.float32)

        # 🔥 negative random
        neg_idx = np.random.randint(0, len(self.t))
        n = torch.tensor(self.t[neg_idx], dtype=torch.float32)

        return r, p, n


dataset = PairDataset(report_seqs, track_seqs, pairs)

In [ ]:
# %%
loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=64,
    shuffle=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

net = TrajTransformer().to(device)
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)

In [ ]:
# %%
for ep in range(20):
    total = 0

    for r, p, n in loader:
        r = r.to(device)
        p = p.to(device)
        n = n.to(device)

        optimizer.zero_grad()

        e_r = net(r)
        e_p = net(p)
        e_n = net(n)

        sim_pos = F.cosine_similarity(e_r, e_p)
        sim_neg = F.cosine_similarity(e_r, e_n)

        loss = F.relu(sim_neg - sim_pos + 0.2).mean()

        loss.backward()
        optimizer.step()

        total += loss.item()

    print(f"Epoch {ep+1}: {total/len(loader):.4f}")

In [ ]:
# %%
def match_topk(net, report_seqs, track_seqs, sub_tracks, top_k=5):
    net.eval()

    with torch.no_grad():
        r = torch.tensor(report_seqs, dtype=torch.float32).to(device)
        t = torch.tensor(track_seqs, dtype=torch.float32).to(device)

        E_r = net(r)
        E_t = net(t)

        sim = torch.matmul(E_r, E_t.T)

    results = {}

    for i in range(len(report_seqs)):
        row = sim[i].cpu().numpy()
        idxs = np.argsort(-row)[:top_k]

        results[i] = [
            {
                "track_id": sub_tracks[j]["track_id"],
                "score": float(row[j])
            }
            for j in idxs
        ]

    return results